## Define Spark Session

In [1]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
from utils import *

root_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
data_dir = os.path.join(root_dir, "data")
spark = SparkSession.builder.appName("etl").config("spark.sql.execution.arrow.pyspark.enabled", "true").getOrCreate()

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/14 23:16:26 WARN Utils: Your hostname, Pradips-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.29.13 instead (on interface en0)
26/06/14 23:16:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/14 23:16:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/14 23:16:27 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 4.1.2


## Create the raw and enriched tables

In [2]:
from Products import Products
from Customers import Customers
from Orders import Orders

products = Products(spark, data_dir)
customers = Customers(spark, data_dir)
orders = Orders(spark, data_dir)

products_df = products.read_products_csv()
customers_df = customers.read_customers_excel()
orders_df = orders.read_orders_json()

products.write_products_raw(products_df)
customers.write_customers_raw(customers_df)
orders.write_orders_raw(orders_df)

products.write_products_enriched(products_df)
customers.write_customers_enriched(customers_df)
orders.write_orders_enriched(orders_df)

products_df = products.read_products_enriched()
customers_df = customers.read_customers_enriched()
orders_df = orders.read_orders_enriched()

/Users/pradipbasak/Projects/pei-de-task/.venv/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:416: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 15.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
26/06/14 23:16:31 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


## order_fact

In [3]:
create_order_fact(customers_df=customers_df, orders_df=orders_df, products_df=products_df, data_dir=data_dir)
order_fact_df = read_order_fact(spark, data_dir)
order_fact_df.show(5, truncate=False)

+--------------+----------+----------+--------------+--------+------+--------+------+-----------+--------------+-------------+---------------+---------------+------------+
|order_id      |order_date|ship_date |ship_mode     |quantity|price |discount|profit|customer_id|customer_name |country      |product_id     |category       |sub_category|
+--------------+----------+----------+--------------+--------+------+--------+------+-----------+--------------+-------------+---------------+---------------+------------+
|CA-2016-122581|2016-08-21|2016-08-25|Standard Class|7       |573.17|0.3     |63.69 |JK-15370   |Jay Kimmel    |United States|FUR-CH-10002961|Furniture      |Chairs      |
|CA-2016-122581|2016-08-21|2016-08-25|Standard Class|7       |573.17|0.3     |63.69 |JK-15370   |Jay Kimmel    |United States|FUR-CH-10002961|Furniture      |Chairs      |
|CA-2017-117485|2017-09-23|2017-09-29|Standard Class|4       |291.96|0.0     |102.19|BD-11320   |Bi1l Donatelli|United States|TEC-AC-1000465

## profit_info

In [4]:
create_profit_info(order_fact_df=order_fact_df, data_dir=data_dir)
profit_info_df = read_profit_info(spark, data_dir)
profit_info_df.filter("coalesce(product_category, product_sub_category) is not null").show(5, truncate=False)

+----+----------------+--------------------+---------------------------+------------+-----------+
|year|product_category|product_sub_category|customer                   |total_profit|order_count|
+----+----------------+--------------------+---------------------------+------------+-----------+
|2017|Furniture       |Bookcases           |             Dorothy Wardle|-32.22      |1          |
|2017|Furniture       |Bookcases           |  Shahi  Collister         |374.63      |1          |
|2017|Furniture       |Bookcases           |Alan Schoenberger          |77.58       |1          |
|2017|Furniture       |Bookcases           |Alice McCarthy             |-64.94      |1          |
|2017|Furniture       |Bookcases           |Anne McFarland             |70.0        |1          |
+----+----------------+--------------------+---------------------------+------------+-----------+
only showing top 5 rows


## aggregated metrics

In [7]:
profit_by_year(spark, order_fact_df).show(5, truncate=False)
profit_by_year_and_category(spark, order_fact_df).show(5, truncate=False)
profit_by_customer(spark, order_fact_df).show(5, truncate=False)
profit_by_customer_and_year(spark, order_fact_df).show(5, truncate=False)

+----+------------+
|year|total_profit|
+----+------------+
|2017|127663.74   |
|2016|68565.57    |
|2015|66289.41    |
|2014|41498.52    |
+----+------------+

+----+----------------+------------+
|year|product_category|total_profit|
+----+----------------+------------+
|2017|Furniture       |3361.76     |
|2017|Office Supplies |45330.39    |
|2017|Technology      |78483.08    |
|2016|Furniture       |7750.15     |
|2016|Office Supplies |35973.6     |
+----+----------------+------------+
only showing top 5 rows
+------------+------------+
|customer    |total_profit|
+------------+------------+
|Frank Hawley|25852.32    |
|Tamara Chand|9010.18     |
|Raymond Buch|6976.36     |
|Sanjit Chand|5839.14     |
|Hunter Lopez|5622.43     |
+------------+------------+
only showing top 5 rows
+----+------------+------------+
|year|customer    |total_profit|
+----+------------+------------+
|2017|Frank Hawley|25388.86    |
|2017|Raymond Buch|6780.9      |
|2017|Patrick Ryan|5555.93     |
|2017|Hu